In [13]:
from typing_extensions import TypedDict, Literal
from typing import List
from langgraph.types import Command
from langgraph.graph import StateGraph, START, END
from langchain.chat_models import init_chat_model
from pydantic import BaseModel

llm = init_chat_model("openai:gpt-4o")

dumb_llm = init_chat_model("openai:gpt-3.5-turbo")
average_llm = init_chat_model("openai:gpt-4o")
smart_llm = init_chat_model("openai:gpt-5-mini-2025-08-07")

In [14]:
class State(TypedDict):
    question: str
    difficulty: list[dict]
    answer: str
    model_used: str

class DifficultyResponse(BaseModel):
    difficulty_level : Literal["easy", "medium", "hard"]


In [ ]:
def dumb_node(state: State):
    response = dumb_llm.invoke(state["question"])
    return {
        "answer": response.content,
        "model_used": "gpt-3.5"
    }

def average_node(state: State):
    response = average_llm.invoke(state["question"])
    return {
        "answer": response.content,
        "model_used": "gpt-4o"
    }

def smart_node(state: State):
    response = smart_llm.invoke(state["question"])
    return {
        "answer": response.content,
        "model_used": "gpt-5-mini"
    }

def assess_difficulty(state: State): 
    structured_llm = llm.with_structured_output(DifficultyResponse)
    response = structured_llm.invoke(
    f"""
    이 질문의 난이도를 평가해 
    질문: {state["question"]}

    - 쉬움: 간단한 사실, 기본정의, 예/아니오 대답에 해당
    - 중간: 설명, 비교 분석이 필요해
    - 어려움: 여러 단계와 깊은 전문성이 필요해
    """)

    difficulty_level = response.difficulty_level

    if difficulty_level == "easy":
        goto = "dumb_node"
    elif difficulty_level == "medium":
        goto = "average_node"
    elif difficulty_level == "hard":
        goto = "smart_node"
    
    return Command(goto=goto, update={"difficulty": difficulty_level})

In [16]:
graph_builder = StateGraph(State)

graph_builder.add_node("dumb_node", dumb_node)
graph_builder.add_node("average_node", average_node)
graph_builder.add_node("smart_node", smart_node)
graph_builder.add_node("assess_difficulty", assess_difficulty, destinations=("dumb_node", "average_node", "smart_node"))

graph_builder.add_edge(START, "assess_difficulty")
graph_builder.add_edge("dumb_node", END)
graph_builder.add_edge("average_node", END)
graph_builder.add_edge("smart_node", END)

graph = graph_builder.compile()

In [18]:
graph.invoke({"question": "한국의 수도는?"})


AttributeError: 'AIMessage' object has no attribute 'contetnt'